# 11 - ABIDE I age and diagnosis: site/ComBat approaches

This notebook compiles the ABIDE validation examples into one interactive pass over two targets:

1. **Age positive control**: `age -> FC`, with sex and mean framewise displacement as nuisance covariates.
2. **Diagnosis contrast**: ASD vs HC, with age, sex, and mean framewise displacement as nuisance covariates.

For both targets, we compare the site-control approaches used across `examples/abide_validation/`:

- **Naive**: no ComBat, no site dummies, no site-stratified permutation.
- **ComBat only**: nuisance-only ComBat, then GLM without site dummies.
- **Strategy E**: no ComBat, but site dummies in the GLM and site-stratified permutation.
- **Strategy D**: nuisance-only ComBat, site dummies in the GLM, and site-stratified permutation.

The production scripts use larger permutation budgets. This notebook uses 50 GPD-accelerated permutations so the whole matrix of examples stays runnable in a tutorial session.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conninfpy import AtlasInfo, analyze
from conninfpy.harmonize import combat_harmonize

plt.rcParams.update({"figure.dpi": 120})

## 1. Load the prepared ABIDE dataset

The prepared file is generated by `examples/abide_validation/prepare_data.py`. It contains Fisher-z-transformed Schaefer-100 connectivity, phenotype covariates, site labels, and ROI network labels.

In [ ]:
candidates = [
    Path("../abide_validation/results/abide_prepared.npz"),
    Path("examples/abide_validation/results/abide_prepared.npz"),
]
DATA_FILE = next((p.resolve() for p in candidates if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "Could not find abide_prepared.npz. Run "
        "`python examples/abide_validation/prepare_data.py` first."
    )

raw = np.load(DATA_FILE, allow_pickle=True)
data = {k: raw[k] for k in raw.files}

Y = data["connectivity_z"]
group = data["group"].astype(float)       # 1 = ASD, 0 = HC
age = data["age"].astype(float)
sex = data["sex"].astype(float)
mean_fd = data["mean_fd"].astype(float)
sites = data["site"]

network_order = list(data["network_order"])
net_labels = np.asarray(data["net_labels"], dtype=int)
networks = [network_order[int(i)] for i in net_labels]
atlas = AtlasInfo(
    labels=[str(x) for x in data["roi_names"]],
    networks=networks,
    source="ABIDE I - Schaefer-100 / Yeo-7",
)

print(f"Loaded: {DATA_FILE}")
print(f"Y shape: {Y.shape}")
print(f"Subjects: {Y.shape[0]}  ROIs: {Y.shape[1]}  sites: {len(np.unique(sites))}")
print(f"ASD: {int(group.sum())}  HC: {int((group == 0).sum())}")

## 2. Site and phenotype balance

ABIDE is multi-site and diagnosis is not perfectly independent of site. The site-aware rows below use the GLM path with `sites=` rather than splitting the cohort into two arrays.

In [ ]:
site_rows = []
for site in np.unique(sites):
    mask = sites == site
    site_rows.append({
        "site": site,
        "ASD": int(np.sum(mask & (group == 1))),
        "HC": int(np.sum(mask & (group == 0))),
        "total": int(np.sum(mask)),
    })
site_df = pd.DataFrame(site_rows).sort_values("total", ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
y = np.arange(len(site_df))
ax.barh(y, site_df["HC"], label="HC", color="#4c78a8")
ax.barh(y, site_df["ASD"], left=site_df["HC"], label="ASD", color="#f58518")
ax.set_yticks(y)
ax.set_yticklabels(site_df["site"], fontsize=8)
ax.set_xlabel("subjects")
ax.set_title("ABIDE subjects per site after QC")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "age_mean": [age[group == 1].mean(), age[group == 0].mean()],
    "age_sd": [age[group == 1].std(), age[group == 0].std()],
    "mean_fd": [mean_fd[group == 1].mean(), mean_fd[group == 0].mean()],
    "male_fraction": [np.mean(sex[group == 1] == 1), np.mean(sex[group == 0] == 1)],
}, index=["ASD", "HC"]).round(3)

## 3. Define targets and approaches

The target definitions mirror the validation scripts:

- `run_age_strategy_d.py`: age as interest, sex and mean FD as nuisance.
- `run_dx_site_glm.py`: diagnosis as interest, age, sex, and mean FD as nuisance.
- `run_dx_strategy_e.py`, `run_dx_combat_only.py`, and `run_naive_baseline.py`: the diagnosis site-control tiers generalized here to both targets.

For a clean method comparison, the notebook keeps each target's non-site nuisance covariates fixed across the four rows; the only thing changing is site/ComBat handling.

In [ ]:
N_PERM = 50
ALPHA = 0.05
COMMON = dict(
    fisher_z=False,
    method="tfnbs",
    e=0.4,
    h=3.0,
    n=10,
    n_permutations=N_PERM,
    acceleration="gpd",
    rng=42,
    use_mp=False,
)

TARGETS = {
    "age": {
        "interest": age,
        "confounds": np.column_stack([sex, mean_fd]),
        "positive": "higher FC with older age",
        "negative": "higher FC with younger age",
    },
    "dx": {
        "interest": group,
        "confounds": np.column_stack([age, sex, mean_fd]),
        "positive": "ASD > HC",
        "negative": "HC > ASD",
    },
}

APPROACHES = [
    ("naive", "Naive: no ComBat, no site GLM"),
    ("combat_only", "ComBat only: no site GLM"),
    ("strategy_e", "Strategy E: site GLM only"),
    ("strategy_d", "Strategy D: ComBat + site GLM"),
]


def run_approach(key, *, interest, confounds):
    diagnostics = {}
    flags = []
    t0 = time.perf_counter()

    if key == "naive":
        out = analyze(
            Y,
            interest=interest,
            confounds=confounds,
            sites=None,
            harmonize=None,
            **COMMON,
        )
    elif key == "combat_only":
        combat = combat_harmonize(Y, sites=sites, preserve=confounds)
        diagnostics = dict(combat.diagnostics)
        out = analyze(
            combat.Y_adjusted,
            interest=interest,
            confounds=confounds,
            sites=None,
            harmonize=None,
            **COMMON,
        )
    elif key == "strategy_e":
        out = analyze(
            Y,
            interest=interest,
            confounds=confounds,
            sites=sites,
            harmonize=None,
            **COMMON,
        )
        diagnostics = out.combat_diagnostics or {}
        flags = out.flags
    elif key == "strategy_d":
        out = analyze(
            Y,
            interest=interest,
            confounds=confounds,
            sites=sites,
            harmonize="nuisance_only",
            **COMMON,
        )
        diagnostics = out.combat_diagnostics or {}
        flags = out.flags
    else:
        raise ValueError(key)

    elapsed = time.perf_counter() - t0
    return out, diagnostics, flags, elapsed

## 4. Run age and diagnosis across all approaches

`positive` and `negative` are interpreted per target. For age, positive means higher FC with older age. For diagnosis, positive means ASD > HC.

In [ ]:
results = {}
rows = []

for target_name, spec in TARGETS.items():
    for approach_key, approach_label in APPROACHES:
        print(f"Running {target_name:>3s} | {approach_key}")
        out, diagnostics, flags, elapsed = run_approach(
            approach_key,
            interest=spec["interest"],
            confounds=spec["confounds"],
        )
        results[(target_name, approach_key)] = out
        nsig = out.inference.n_significant(alpha=ALPHA)
        iu = np.triu_indices(Y.shape[1], 1)
        rows.append({
            "target": target_name,
            "approach": approach_label,
            "positive_meaning": spec["positive"],
            "negative_meaning": spec["negative"],
            "positive_edges": nsig["positive"],
            "negative_edges": nsig["negative"],
            "min_p_positive": float(out["positive"][iu].min()),
            "min_p_negative": float(out["negative"][iu].min()),
            "combat_ratio_after_before": diagnostics.get("between_site_variance_ratio_after_over_before", np.nan),
            "site_strata": bool(getattr(out.inference, "strata_provided", False)),
            "seconds": elapsed,
            "flags": "; ".join(flags),
        })

summary = pd.DataFrame(rows)
summary[[
    "target", "approach", "positive_edges", "negative_edges",
    "min_p_positive", "min_p_negative", "combat_ratio_after_before",
    "site_strata", "seconds",
]].round(4)

## 5. Compare significant-edge counts

These counts are a tutorial-scale quick run. The validation scripts use larger permutation budgets before writing paper tables.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, target_name in zip(axes, TARGETS):
    sub = summary[summary["target"] == target_name].copy()
    labels = [a.split(":")[0] for a in sub["approach"]]
    x = np.arange(len(sub))
    ax.bar(x - 0.18, sub["positive_edges"], width=0.36, label="positive", color="#4c78a8")
    ax.bar(x + 0.18, sub["negative_edges"], width=0.36, label="negative", color="#f58518")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_title(target_name)
    ax.set_ylabel("significant edges")
    ax.grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

## 6. Network-pair heatmaps for age and diagnosis

This reproduces the spirit of the diagnosis methodology quadrant plots, but applies the same site/ComBat grid to both targets.

In [ ]:
def block_count_matrix(result):
    df = result.significant_edges(atlas=atlas, alpha=ALPHA, sort="p")
    n_nets = len(network_order)
    mat = np.zeros((n_nets, n_nets), dtype=int)
    if df.empty:
        return mat
    for _, row in df.iterrows():
        ni = int(net_labels[int(row["roi_i"])])
        nj = int(net_labels[int(row["roi_j"])])
        mat[ni, nj] += 1
        if ni != nj:
            mat[nj, ni] += 1
    return mat

mats = {(target, key): block_count_matrix(results[(target, key)])
        for target in TARGETS for key, _ in APPROACHES}
vmax = max(int(mat.max()) for mat in mats.values()) or 1

fig, axes = plt.subplots(2, 4, figsize=(14, 7), constrained_layout=True)
for r, target_name in enumerate(TARGETS):
    for c, (approach_key, approach_label) in enumerate(APPROACHES):
        ax = axes[r, c]
        mat = mats[(target_name, approach_key)]
        im = ax.imshow(mat, cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(f"{target_name}: {approach_label.split(':')[0]}", fontsize=9)
        ax.set_xticks(range(len(network_order)))
        ax.set_yticks(range(len(network_order)))
        if r == 1:
            ax.set_xticklabels(network_order, rotation=45, ha="right", fontsize=6)
        else:
            ax.set_xticklabels([])
        if c == 0:
            ax.set_yticklabels(network_order, fontsize=6)
        else:
            ax.set_yticklabels([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, label="significant edges")
plt.show()

## 7. Top Strategy D edges for each target

Strategy D is the primary site-aware recipe used by the ABIDE validation scripts: nuisance-only ComBat, site dummies in the GLM, and site-stratified permutation.

In [ ]:
top_tables = []
for target_name in TARGETS:
    df = results[(target_name, "strategy_d")].significant_edges(
        atlas=atlas,
        alpha=ALPHA,
        sort="network_pair",
        top_k=8,
    ).copy()
    if not df.empty:
        df.insert(0, "target", target_name)
        top_tables.append(df)

if top_tables:
    top_edges = pd.concat(top_tables, ignore_index=True)
    cols = [
        "target", "tail", "network_pair", "roi_i_name", "roi_j_name",
        "t_signed", "p_positive", "p_negative", "p_min",
    ]
    display(top_edges[cols].round({
        "t_signed": 3,
        "p_positive": 4,
        "p_negative": 4,
        "p_min": 4,
    }))
else:
    print("No Strategy D significant edges in this quick run.")

## Takeaways

- The age row is the positive-control side of the ABIDE validation story.
- The diagnosis row is the observational ASD vs HC contrast.
- The four columns separate the roles of ComBat, site dummies, and site-stratified permutation.
- Strategy D is the primary recipe in the validation suite because ComBat excludes the tested variable, site dummies remain in the GLM, and permutations respect site exchangeability blocks.
- Increase `N_PERM` or run the scripts in `examples/abide_validation/` for production-quality tables.